# FINPLE Monthly Metrics One-Click Pipeline

Step 114-2D offline rolling metrics and review-only overlay Colab workflow. The notebook exposes five operator sections and delegates calculation, source adapter selection, raw daily normalization, rolling CAGR, and audit generation to the repository module. In a fresh Colab runtime, the public FINPLE repository is cloned automatically. If cloning is unavailable, the notebook falls back to an uploaded Step 114-2D execution package ZIP. No private GitHub token is required.

In [ ]:
# 정상 운영에서는 AS_OF_INCLUDED만 변경하세요.
AS_OF_INCLUDED = "2026-07-22"
HISTORY_YEARS = 20
PARTIAL_MONTH_POLICY = "exclude_from_metrics"
USE_GOOGLE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/FINPLE/monthly-metrics"


## 1. Check settings

For normal monthly operation, change only `AS_OF_INCLUDED`. The default 20-year US/KR collection outputs and One-Click package are read from and written to the date-specific Google Drive folder. Set `USE_GOOGLE_DRIVE=False` to use the equivalent `/content/finple-monthly-metrics/<date>` local path. The public clone remains the default bootstrap and ZIP upload remains checkout fallback only.

In [ ]:
!rm -rf /content/FINPLE
!git clone --depth 1 https://github.com/vip930sw/FINPLE.git /content/FINPLE
%cd /content/FINPLE

In [ ]:
from pathlib import Path
from datetime import date, datetime
import hashlib
import os
import shutil
import subprocess
import sys
import zipfile

CLONE_ROOT = Path("/content/FINPLE")
COLLECTION_REF = "codex/step114-2y-one-click-full-universe-app-preview"

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "metrics_pipeline").exists() and (candidate / "data" / "fixtures" / "monthly-metrics").exists():
            return candidate
    return None

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

REPO_ROOT = find_repo_root(CLONE_ROOT) if CLONE_ROOT.exists() else None
if REPO_ROOT is None:
    REPO_ROOT = find_repo_root(Path.cwd())
bootstrap_source = "git_clone" if REPO_ROOT is not None and REPO_ROOT == CLONE_ROOT else ("existing_checkout" if REPO_ROOT is not None else None)
uploaded_package_sha256 = None

if REPO_ROOT is None:
    try:
        from google.colab import files
        print("Public GitHub clone did not produce a FINPLE repository checkout. Upload the Step 114-2D execution package ZIP fallback.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No execution package ZIP uploaded.")
        zip_name = next(iter(uploaded))
        uploaded_package_sha256 = sha256_file(zip_name)
        extract_root = Path("/content/finple_step114_2d_execution_package")
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_name) as archive:
            archive.extractall(extract_root)
        REPO_ROOT = find_repo_root(extract_root)
        bootstrap_source = "uploaded_execution_package"
    except ImportError as exc:
        raise RuntimeError("Repository files were not found and the Colab upload helper is unavailable.") from exc

if REPO_ROOT is None:
    raise RuntimeError("FINPLE repository bootstrap failed.")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

def run_visible(command, label, cwd):
    resolved_cwd = Path(cwd).resolve()
    print({'label': label, 'command': [str(part) for part in command], 'cwd': str(resolved_cwd)}, flush=True)
    try:
        result = subprocess.run(
            command, cwd=resolved_cwd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False
        )
    except OSError as exc:
        print({'returnCode': None}, flush=True)
        print('--- stdout ---\n', flush=True)
        print(f'--- stderr ---\n{exc}', flush=True)
        raise RuntimeError(f'{label} could not start') from exc
    print({'returnCode': result.returncode}, flush=True)
    print('--- stdout ---', flush=True)
    print(result.stdout or '', end='' if (result.stdout or '').endswith('\n') else '\n', flush=True)
    print('--- stderr ---', flush=True)
    print(result.stderr or '', end='' if (result.stderr or '').endswith('\n') else '\n', flush=True)
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit status {result.returncode}')
    return result

if bootstrap_source == "git_clone":
    run_visible(["git", "fetch", "--depth", "1", "origin", COLLECTION_REF], "fetch requested ref", REPO_ROOT)
    run_visible(["git", "checkout", "--detach", "FETCH_HEAD"], "detached FETCH_HEAD checkout", REPO_ROOT)
repo_sha = None
git_dir = REPO_ROOT / ".git"
if git_dir.exists():
    repo_sha = run_visible(["git", "rev-parse", "HEAD"], "repository HEAD", REPO_ROOT).stdout.strip()

PREPARE_MODULE = "scripts.prepare_monthly_metrics_candidate_inputs"
run_visible([sys.executable, "-m", PREPARE_MODULE, "--help"], "candidate prepare module CLI preflight", REPO_ROOT)

RUN_CANDIDATE_MODE = True
PREPARE_FROM_COLLECTED_RAW = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
RUN_ROOT = Path(DRIVE_ROOT if USE_GOOGLE_DRIVE else "/content/finple-monthly-metrics") / AS_OF_INCLUDED
RUN_PATHS = {name: RUN_ROOT / folder for name, folder in {"smoke": "smoke", "chunks": "chunks", "combined": "combined", "validation": "validation", "one_click": "one-click"}.items()}
for path in [RUN_ROOT, *RUN_PATHS.values()]:
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "metric_base_date": AS_OF_INCLUDED,
    "market_scope": ["US", "KR"],
    "selected_cagr_policy": "rolling_median_all_markets",
    "current_price_display": False,
    "total_return_cagr_mode": "reference_only",
    "output_version": AS_OF_INCLUDED.replace("-", "_"),
    "input_mode": "fixture",  # fixture | manual_upload | public_source_fixture
    "input_dir": str(REPO_ROOT / "data" / "fixtures" / "monthly-metrics"),
    "output_dir": str(RUN_PATHS["one_click"] / "fixture"),
    "deterministic_fixture": True,
    "random_seed": 1142,
    "partial_month_policy": PARTIAL_MONTH_POLICY,
}

CANDIDATE_CONFIRMATION_TOKEN = "RUN_STEP114_2Y_INTERNAL_PREVIEW"
CANDIDATE_REQUIRED_FILES = [
    "candidate_asset_master.csv",
    "benchmark_map.csv",
    "raw_daily_prices.csv",
    "source_declaration.json",
    "operator_submission_manifest.json",
]
COLLECTED_RAW_REQUIRED_FILES = [
    "us_raw_daily_prices.csv",
    "kr_raw_daily_prices.csv",
    "kr_price_metrics_overlay.csv",
]
ATTEMPT_ID = f"step114-2y-{datetime.utcnow().strftime('%Y%m%dT%H%M%S%f')}"
CANDIDATE_UPLOAD_ROOT = RUN_PATHS["validation"] / "candidate-input-attempts" / ATTEMPT_ID
CANDIDATE_REPORT_ROOT = RUN_PATHS["validation"] / "attempt-reports" / ATTEMPT_ID
COLLECTED_RAW_ROOT = RUN_PATHS["combined"]
CANDIDATE_OUTPUT_ROOT = RUN_PATHS["one_click"] / "attempts" / ATTEMPT_ID

CANDIDATE_CONFIG = {
    "metric_base_date": AS_OF_INCLUDED,
    "market_scope": ["US", "KR"],
    "output_version": f"{AS_OF_INCLUDED.replace('-', '_')}_step114_2y_internal_preview",
    "input_mode": "manual_upload_candidate",
    "input_dir": str(CANDIDATE_UPLOAD_ROOT),
    "output_dir": str(CANDIDATE_OUTPUT_ROOT),
    "validation_date": date.today().isoformat(),
    "internal_preview_review_only": True,
    "partial_month_policy": PARTIAL_MONTH_POLICY,
}

print(f"Repository root: {REPO_ROOT}")
print(f"Bootstrap source: {bootstrap_source}")
if repo_sha:
    print(f"Repository SHA: {repo_sha}")
elif uploaded_package_sha256:
    print(f"Execution package SHA-256: {uploaded_package_sha256}")
else:
    print("Repository SHA: unavailable")
print({"requestedAsOfIncluded": AS_OF_INCLUDED, "metricBaseDate": AS_OF_INCLUDED, "partialMonthPolicy": PARTIAL_MONTH_POLICY, "historyYears": HISTORY_YEARS, "useGoogleDrive": USE_GOOGLE_DRIVE, "runRoot": str(RUN_ROOT)})
CONFIG

## 2. Check inputs

Normal candidate mode reads the two combined raw-daily files plus the KR metrics resolution file directly from the date-specific `combined` folder. No browser upload is required when the US/KR collectors used the same settings. This One-Click notebook never calls an external provider.


In [ ]:
active_config = CANDIDATE_CONFIG if RUN_CANDIDATE_MODE else CONFIG
input_dir = Path(active_config["input_dir"])
if RUN_CANDIDATE_MODE:
    if active_config.get("input_mode") != "manual_upload_candidate":
        raise ValueError("Candidate mode requires input_mode=manual_upload_candidate")
    if CANDIDATE_CONFIRMATION_TOKEN != "RUN_STEP114_2Y_INTERNAL_PREVIEW":
        raise RuntimeError("Set CANDIDATE_CONFIRMATION_TOKEN to RUN_STEP114_2Y_INTERNAL_PREVIEW after reviewing the review-only candidate contract.")
    print("Step 114-2Y contract: no provider calls in One-Click; unapproved license/publication remains review-only; productionPublishReady=false; appExportApproved=false.")
    retry_targets = [
        (input_dir, RUN_PATHS["validation"] / "candidate-input-attempts"),
        (Path(active_config["output_dir"]), RUN_PATHS["one_click"] / "attempts"),
        (CANDIDATE_REPORT_ROOT, RUN_PATHS["validation"] / "attempt-reports"),
    ]
    for target, expected_parent in retry_targets:
        if target.exists():
            if target.resolve().parent != expected_parent.resolve() or target.name != ATTEMPT_ID:
                raise RuntimeError(f"Unsafe One-Click attempt reset target: {target}")
            shutil.rmtree(target)
    input_dir.mkdir(parents=True, exist_ok=False)
    CANDIDATE_REPORT_ROOT.mkdir(parents=True, exist_ok=False)
    if PREPARE_FROM_COLLECTED_RAW:
        collected_names = sorted(path.name for path in COLLECTED_RAW_ROOT.iterdir() if path.is_file() and path.name in COLLECTED_RAW_REQUIRED_FILES)
        if collected_names != sorted(COLLECTED_RAW_REQUIRED_FILES):
            raise RuntimeError(f"Collected combined files are incomplete in {COLLECTED_RAW_ROOT}: expected {sorted(COLLECTED_RAW_REQUIRED_FILES)}, got {collected_names}")
        reconciliation_report = CANDIDATE_REPORT_ROOT / "candidate_input_reconciliation.json"
        run_visible([
            sys.executable, "-m", PREPARE_MODULE,
            "--universe", str(REPO_ROOT / "src" / "data" / "tickers" / "finple_app_candidates_6000_balanced_v1.csv"),
            "--us-raw", str(COLLECTED_RAW_ROOT / "us_raw_daily_prices.csv"),
            "--kr-raw", str(COLLECTED_RAW_ROOT / "kr_raw_daily_prices.csv"),
            "--kr-metrics", str(COLLECTED_RAW_ROOT / "kr_price_metrics_overlay.csv"),
            "--output-dir", str(input_dir),
            "--report", str(reconciliation_report),
            "--metric-base-date", CANDIDATE_CONFIG["metric_base_date"],
            "--as-of-included", AS_OF_INCLUDED,
            "--partial-month-policy", PARTIAL_MONTH_POLICY,
            "--submission-id", ATTEMPT_ID,
        ], "prepare monthly metrics candidate inputs", REPO_ROOT)
        print(reconciliation_report.read_text(encoding="utf-8"))
    else:
        required_inputs = list(CANDIDATE_REQUIRED_FILES)
        print("Required prepared files:", required_inputs)
        from google.colab import files
        uploaded = files.upload()
        uploaded_names = sorted(uploaded.keys())
        if uploaded_names != sorted(required_inputs):
            raise RuntimeError(f"Upload exactly these files and no extras: {sorted(required_inputs)}; got {uploaded_names}")
        for file_name, content in uploaded.items():
            if file_name not in required_inputs or "/" in file_name or "\\" in file_name or ".." in Path(file_name).parts:
                raise RuntimeError(f"Unsafe candidate upload file name: {file_name}")
            (input_dir / file_name).write_bytes(content)
    required_inputs = list(CANDIDATE_REQUIRED_FILES)
    if sorted(path.name for path in input_dir.iterdir() if path.is_file()) != sorted(required_inputs):
        raise RuntimeError("Candidate upload directory does not match the exact required file set.")
    print(f"Candidate files uploaded outside Git checkout: {input_dir}")
else:
    required_inputs = ["candidates.csv", "benchmark_map.csv", "monthly_prices.csv"]
    if active_config["input_mode"] == "fixture":
        required_inputs.append("raw_daily_prices.csv")
    elif active_config["input_mode"] == "manual_upload":
        required_inputs.append(active_config.get("manual_upload_file", "manual_upload_raw_daily_prices.csv"))
    elif active_config["input_mode"] == "public_source_fixture":
        required_inputs.append(active_config.get("public_source_fixture_file", "public_source_fixture_prices.csv"))
    else:
        raise ValueError(f"Unsupported offline input_mode: {active_config['input_mode']}")

for file_name in required_inputs:
    path = input_dir / file_name
    print(f"{file_name}: {'OK' if path.exists() else 'MISSING'}")


## 3. Run pipeline

Run the single repository entrypoint. Fixture mode remains the default. Candidate mode uses the explicit offline production-candidate entrypoint and still returns `productionPublishReady=false` and `appExportApproved=false`.


In [ ]:
from scripts.metrics_pipeline import run_finple_monthly_metrics_pipeline, run_finple_production_candidate_package

if RUN_CANDIDATE_MODE:
    RESULT = run_finple_production_candidate_package(CANDIDATE_CONFIG)
else:
    RESULT = run_finple_monthly_metrics_pipeline(CONFIG)


## 4. Review summary

Review counts, rolling window counts, adapter summary, manifest versions, review-only overlay paths, and generated paths before using any output.

In [ ]:
print("FINPLE Monthly Metrics Pipeline Complete")
print(f"Mode: {'candidate' if RUN_CANDIDATE_MODE else 'fixture'}")
print(f"Fixture package ready: {RESULT['fixturePackageReady']}")
print(f"Candidate package ready: {RESULT.get('candidatePackageReady', False)}")
print(f"Production publish ready: {RESULT['productionPublishReady']}")
print(f"App export approved: {RESULT['appExportApproved']}")
if RUN_CANDIDATE_MODE:
    print(f"Candidate package id: {RESULT.get('candidatePackageId', '')}")
    print(f"Blocking issues: {RESULT.get('blockingIssueCount', 0)}")
    print(f"Warning issues: {RESULT.get('warningIssueCount', 0)}")
    for key in ["requestedAsOfIncluded", "actualLastPriceDate", "metricDataThroughMonth", "partialFinalMonthDetected", "partialFinalMonthExcluded", "partialMonthPolicy"]:
        print(f"{key}: {RESULT.get(key, '')}")
    for key in ["metricsOutputCsv", "selectedMetricsCsv", "reviewRequiredCsv", "monthlyReturnsCsv", "normalizedMonthEndCsv", "timeseriesAuditCsv", "usReviewOverlayCsv", "krReviewOverlayCsv", "auditHtml", "manifestJson", "zipPackage"]:
        print(f"{key}: {RESULT['outputs'].get(key, '')}")
else:
    print(f"Base date: {RESULT['metricBaseDate']}")
    print(f"Pipeline version: {RESULT['pipelineVersion']}")
    print(f"Source adapter summary: {RESULT['outputs']['sourceAdapterSummaryJson']}")
    print(f"Source adapter checkpoint: {RESULT['outputs']['sourceAdapterCheckpointJson']}")
    print(f"US review-only overlay: {RESULT['outputs']['usReviewOverlayCsv']}")
    print(f"KR review-only overlay: {RESULT['outputs']['krReviewOverlayCsv']}")
    for key, value in RESULT["summary"].items():
        print(f"{key}: {value}")
print(f"Output ZIP: {RESULT['outputs'].get('zipPackage', '')}")


## 5. Download output ZIP and optional cleanup

Download the ZIP after checking the audit report and manifest. Candidate ZIPs are review-only packages, not production app approval. For candidate mode, set `CLEANUP_CANDIDATE_FILES=True` after confirming download to delete uploaded raw files and generated outputs from the Colab runtime.

In [ ]:
CLEANUP_CANDIDATE_FILES = False

DOWNLOAD_OUTPUT_ZIP = False
zip_path = Path(RESULT["outputs"]["zipPackage"])
print(zip_path)

if DOWNLOAD_OUTPUT_ZIP:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Browser download is disabled by default. The One-Click ZIP remains at the path above.")

if RUN_CANDIDATE_MODE and CLEANUP_CANDIDATE_FILES:
    for target in [Path(CANDIDATE_CONFIG["input_dir"]), Path(CANDIDATE_CONFIG["output_dir"])]:
        if target.exists():
            shutil.rmtree(target)
        print(f"{target}: {'DELETED' if not target.exists() else 'STILL_EXISTS'}")
elif RUN_CANDIDATE_MODE:
    print("Candidate cleanup not run. Set CLEANUP_CANDIDATE_FILES=True after confirming ZIP download.")
